# 04b - SCD Type 2: Merge Logic (Recurring, Idempotent)

**Project:** Retail Analytics & Product Dimension History

## Safe to run on a schedule
This notebook contains ONLY the generic, reusable SCD Type 2 merge logic -
no hardcoded product IDs, no table resets. Proven idempotent via
DESCRIBE HISTORY (a second run of the same merge affects 0 rows). In a
genuinely live production system, this notebook would run against whatever
new extract had actually landed since the last run - if nothing changed,
it correctly does nothing, which is the right behavior for a recurring job,
not a limitation of this demo.

## Prerequisite
Requires 04a_retail_scd2_backfill to have been run at least once, manually,
before this notebook is ever scheduled.

## Table updated
- `main.retail_analytics.silver_dim_products_scd2`

In [0]:
from pyspark.sql.functions import col

new_product_extract_df = (
    spark.table("main.retail_analytics.bronze_products_day2_extract")
    .dropDuplicates(["product_id"])
)
new_product_extract_df.createOrReplaceTempView("new_product_extract")
print("Rows in new extract:", new_product_extract_df.count())

In [0]:
%sql
MERGE INTO main.retail_analytics.silver_dim_products_scd2 AS target
USING new_product_extract AS source
ON target.product_id = source.product_id AND target.is_current = true
WHEN MATCHED AND (
  target.category != source.category
  OR target.subcategory != source.subcategory
  OR target.unit_price != source.unit_price
  OR target.cost_price != source.cost_price
)
THEN UPDATE SET
  target.is_current = false,
  target.effective_end_date = current_date()

In [0]:
%sql
MERGE INTO main.retail_analytics.silver_dim_products_scd2 AS target
USING new_product_extract AS source
ON target.product_id = source.product_id
WHEN NOT MATCHED BY SOURCE AND target.is_current = true
THEN UPDATE SET
  target.is_current = false,
  target.effective_end_date = current_date()

In [0]:
%sql
MERGE INTO main.retail_analytics.silver_dim_products_scd2 AS target
USING new_product_extract AS source
ON target.product_id = source.product_id AND target.is_current = true
WHEN NOT MATCHED THEN INSERT (
  product_sk, product_id, product_name, category, subcategory, unit_price, cost_price,
  effective_start_date, effective_end_date, is_current
)
VALUES (
  uuid(), source.product_id, source.product_name, source.category, source.subcategory,
  source.unit_price, source.cost_price, current_date(), NULL, true
)

In [0]:
%sql
SELECT COUNT(*) AS total_rows FROM main.retail_analytics.silver_dim_products_scd2;







In [0]:
%sql
SELECT COUNT(*) AS current_products FROM main.retail_analytics.silver_dim_products_scd2 WHERE is_current = true;

In [0]:
%sql
SELECT product_id, COUNT(*) AS current_version_count
FROM main.retail_analytics.silver_dim_products_scd2
WHERE is_current = true
GROUP BY product_id
HAVING COUNT(*) != 1;

In [0]:
%sql
SELECT COUNT(*) AS p015_current_count
FROM main.retail_analytics.silver_dim_products_scd2
WHERE product_id = 'P015' AND is_current = true;